# Assignment 4 — Sequential knowledge editing experiment (Kaggle runner)

**Before running:** Settings → Accelerator **GPU T4 ×2**, Internet **On** (requires a phone-verified Kaggle account).

**Order of work**
1. `MODE = "setup"` → prefilter + covariance statistics for your model (≈20–40 min).
2. `MODE = "pilot"` → 5 edits per method on the **dev** split, prints seconds/edit. Send these numbers to the team → fill `n_max` in the protocol freeze form → **upload the form to Moodle**.
3. `MODE = "main"` → final runs. Use *Save Version → Save & Run All (Commit)* so it runs in the background (you can close the browser).
   Each session stops itself after `HOURS` hours; to continue, add this notebook's previous output as input (*Add Input → Your Work → this notebook*) and commit again — runs resume from their last checkpoint.
4. `MODE = "analysis"` (any account, after all results are merged) → error-analysis CSV + tables + figures.


In [ ]:
# ======== EDIT THESE ========
REPO_URL = "https://github.com/YOUR_GITHUB/ke-sequential-editing.git"   # your team's repo
BRANCH   = "main"
MODEL    = "gpt2-xl"          # "gpt2-xl" or "llama3.2-3b"  (one model per Kaggle account)
MODE     = "setup"            # "setup" | "pilot" | "main" | "analysis"
HOURS    = 11.0               # stop each GPU queue after this many hours (Kaggle limit is 12 h)
# Final-run job lists (model:method:seed[:single]); split across the two T4 GPUs
JOBS_GPU0 = [f"{MODEL}:ROME:0", f"{MODEL}:ROME:1", f"{MODEL}:ROME:2", f"{MODEL}:FT:0", f"{MODEL}:FT:1", f"{MODEL}:FT:2"]
JOBS_GPU1 = [f"{MODEL}:MEMIT:0", f"{MODEL}:MEMIT:1", f"{MODEL}:MEMIT:2",
             f"{MODEL}:none:0", f"{MODEL}:none:1", f"{MODEL}:none:2",
             f"{MODEL}:ROME:0:single", f"{MODEL}:MEMIT:0:single", f"{MODEL}:FT:0:single"]


In [ ]:
import os, subprocess, glob, json
W = "/kaggle/working/ke-seq"
if not os.path.exists(W):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, W], check=True)
else:
    subprocess.run(["git", "-C", W, "pull"], check=False)
os.chdir(W)
print(subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Versions tested by EasyEdit; torch is NOT reinstalled (Kaggle's CUDA build is kept, avoiding the A3 crash)
!pip install -q "transformers==5.5.4" "datasets==4.8.5" "huggingface_hub==1.17.0"
!python -c "import torch, transformers, datasets; print(torch.__version__, torch.version.cuda, transformers.__version__, datasets.__version__, torch.cuda.device_count())"


In [ ]:
# Restore results from a previous session (if its output was added as input)
prev = [p for p in glob.glob("/kaggle/input/*/ke-seq/results") + glob.glob("/kaggle/input/*/*/ke-seq/results")]
for p in prev:
    print("restoring from", p)
    !rsync -a "{p}/" results/
prev_stats = [p for p in glob.glob("/kaggle/input/*/ke-seq/data/stats") + glob.glob("/kaggle/input/*/*/ke-seq/data/stats")]
for p in prev_stats:
    !rsync -a "{p}/" data/stats/
prev_ppl = glob.glob("/kaggle/input/*/ke-seq/data/processed/ppl_passages.jsonl") + glob.glob("/kaggle/input/*/*/ke-seq/data/processed/ppl_passages.jsonl")
if prev_ppl and not os.path.exists("data/processed/ppl_passages.jsonl"):
    !cp "{prev_ppl[0]}" data/processed/
!cd data/processed && sha256sum -c SHA256SUMS

In [ ]:
if MODE == "setup":
    !python src/prefilter.py --model {MODEL}
    !python src/compute_cov.py --model {MODEL}
    !sha256sum data/processed/ppl_passages.jsonl

In [ ]:
if MODE == "pilot":   # dev split only - never reported as results
    for m in ["none", "FT", "ROME", "MEMIT"]:
        !python src/run_experiment.py --model {MODEL} --method {m} --seed 0 --split dev --n_max 5 --overwrite 2>&1 | tail -3
    for d in sorted(glob.glob(f"results/{MODEL}/seq/*_dev")):
        print(d, json.load(open(d + "/run_summary.json")))

In [ ]:
if MODE == "main":
    p0 = subprocess.Popen(["python", "src/run_queue.py", "--gpu", "0", "--hours", str(HOURS), "--jobs", *JOBS_GPU0])
    p1 = subprocess.Popen(["python", "src/run_queue.py", "--gpu", "1", "--hours", str(HOURS), "--jobs", *JOBS_GPU1])
    p0.wait(); p1.wait()

In [ ]:
# Progress of every run
for d in sorted(glob.glob(f"results/*/*/*_seed*")):
    prog = json.load(open(d + "/progress.json")) if os.path.exists(d + "/progress.json") else {}
    summ = json.load(open(d + "/run_summary.json")) if os.path.exists(d + "/run_summary.json") else {}
    print(f"{d:55s} done={prog.get('done')}  {summ}")

In [ ]:
if MODE == "analysis":
    for m in ["gpt2-xl", "llama3.2-3b"]:
        if os.path.isdir(f"results/{m}/seq"):
            !python src/error_export.py --model {m}
    !python src/analyze.py
    !cat results/analysis/main_results.md